<a href="https://colab.research.google.com/github/Mustafizb24/gold-directional-forecasting/blob/main/03_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

#Loading the cleaned features dataset from notebook 1

df = pd.read_csv('/content/drive/MyDrive/gold_features.csv', index_col=0, parse_dates = True)

#Checking size, start date and the percentage of up days
print(df.shape)
print(df.index.min())
print(df["target"].mean())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(3974, 12)
2010-03-16 00:00:00
0.5266733769501761


In [3]:
from sklearn.model_selection import TimeSeriesSplit #Main tool for walk-forward splits

# The 8 features used as model inputs (raw prices are left out)
feature_cols = ["gold_ret", "dxy_ret", "yield_chg", "gold_ret_lag1", "gold_ret_lag2", "gold_ma20_gap", "gold_ma50_gap", "gold_vol20"]

# x = inputs, y = what we're predicting
x = df[feature_cols]
y = df["target"]

#Walk-forward: 5 splits, each one trains on all past data and tests on the next block of time
tscv = TimeSeriesSplit(n_splits = 5)

# Checking the date range of each split
for i, (train_idx, test_idx) in enumerate(tscv.split(x)):
  train_start = x.index[train_idx[0]].date()
  train_end = x.index[train_idx[-1]].date()
  test_start = x.index[test_idx[0]].date()
  test_end = x.index[test_idx[-1]].date()

  print("Split", i+1, " |  train:", train_start, "to", train_end,
        " |  test:", test_start, "to", test_end )

Split 1  |  train: 2010-03-16 to 2012-10-31  |  test: 2012-11-01 to 2015-06-19
Split 2  |  train: 2010-03-16 to 2015-06-19  |  test: 2015-06-22 to 2018-02-08
Split 3  |  train: 2010-03-16 to 2018-02-08  |  test: 2018-02-09 to 2020-09-25
Split 4  |  train: 2010-03-16 to 2020-09-25  |  test: 2020-09-28 to 2023-05-15
Split 5  |  train: 2010-03-16 to 2023-05-15  |  test: 2023-05-16 to 2025-12-30


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

#Pipeline: scaling features first, then run logistic regression
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

print(lr_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('lr', LogisticRegression(max_iter=1000))])
